In [1]:
from pathlib import Path
PROJECT = Path('/home/featurize/work/biological_prior_validation')
CONFIG = {
    'project_root': str(PROJECT),
    'run_path': str(PROJECT/'data/splits/step2_q05_seed20261008'),
    'seed': 20261008,
    'mu_floor': 1e-8,                     
    'relative_min_denominator': 1e-12,   
    'batch_cells': 256,                 
}
RUN_FINAL_EVALUATION = False 
METHOD_SPECS = []             
print('Run:', CONFIG['run_path'])


Run: /home/featurize/work/biological_prior_validation/data/splits/step2_q05_seed20261008


In [2]:
SOURCE_CODE = '"""Fixed count-level evaluator; fitting/calibration accepts no final target information."""\nfrom pathlib import Path\nimport json, hashlib, sys, platform, subprocess\nimport numpy as np\nimport pandas as pd\nimport scipy\nimport scipy.sparse as sp\nfrom scipy.special import kl_div\n\nENGINE_SHA=hashlib.sha256(Path(__file__).read_bytes()).hexdigest() if \'__file__\' in globals() else None\nLINES=[\'A172\',\'T98G\',\'U87MG\'];DRUGS=[\'Vehicle\',\'Lapatinib\',\'Nintedanib\',\'Trametinib\',\'ZSTK474\']\n\ndef digest(p):\n    h=hashlib.sha256()\n    with Path(p).open(\'rb\') as f:\n        for b in iter(lambda:f.read(1048576),b\'\'):h.update(b)\n    return h.hexdigest()\n\ndef save_json(p,obj):\n    p=Path(p);t=p.with_name(p.name+\'.tmp\');t.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str));t.replace(p)\n\ndef _fingerprint(obj):return hashlib.sha256(json.dumps(obj,sort_keys=True,separators=(\',\',\':\')).encode()).hexdigest()\n\ndef _json(p):return json.loads(Path(p).read_text())\n\ndef inverse_log1p_to_B(xhat,library_A,q,scale=10000):\n    """No B-library fitting, no selected-gene renormalization, no upper clipping."""\n    x=np.asarray(xhat,dtype=np.float64);lib=np.asarray(library_A,dtype=np.float64)\n    if x.ndim!=2 or lib.shape!=(len(x),) or not np.isfinite(x).all() or not np.isfinite(lib).all() or (lib<=0).any():raise ValueError(\'Invalid prediction or positive-A-library inputs\')\n    if not 0<q<1 or not np.isfinite(scale) or scale<=0:raise ValueError(\'Invalid q/normalization scale\')\n    with np.errstate(over=\'raise\',invalid=\'raise\'):\n        mu=np.expm1(np.maximum(x,0))*lib[:,None]/scale*(1-q)/q\n    if not np.isfinite(mu).all():raise ValueError(\'Nonfinite count predictions; no hidden upper cap\')\n    return mu\n\ndef score_rows(B,prediction,library_A,library_B,q,scale,mu_floor,prediction_type=\'log_normalized\'):\n    """Mean is formed later with exact pair counts. Includes zero target entries."""\n    b=B.toarray().astype(np.float64) if sp.issparse(B) else np.asarray(B,dtype=np.float64)\n    pred=np.asarray(prediction,dtype=np.float64);la=np.asarray(library_A,dtype=np.float64);lb=np.asarray(library_B,dtype=np.float64)\n    if b.ndim!=2 or pred.shape!=b.shape or la.shape!=(len(b),) or lb.shape!=(len(b),):raise ValueError(\'Evaluation shape mismatch\')\n    if not np.isfinite(b).all() or (b<0).any() or not np.equal(b,np.floor(b)).all():raise ValueError(\'B must be finite nonnegative integer counts\')\n    if not np.isfinite(pred).all() or not np.isfinite(lb).all() or (lb<0).any() or (b.sum(axis=1)>lb+1e-8).any():raise ValueError(\'Invalid predictions or all-gene B library\')\n    if not np.isfinite(la).all() or (la<=0).any() or not 0<q<1 or scale<=0 or not np.isfinite(mu_floor) or mu_floor<=0:raise ValueError(\'Invalid fixed scoring configuration\')\n    if prediction_type==\'log_normalized\':\n        mu=inverse_log1p_to_B(pred,la,q,scale);x_pred=pred\n    elif prediction_type==\'expected_B_counts\':\n        if (pred<0).any():raise ValueError(\'Expected-count predictions must be nonnegative\')\n        mu=pred.copy();x_pred=np.log1p(scale*mu/(la[:,None]*(1-q)/q))\n    else:raise ValueError(\'Unknown prediction type\')\n    floored=mu<mu_floor;mu=np.maximum(mu,mu_floor)\n    with np.errstate(over=\'raise\',invalid=\'raise\'):\n        losses=2*kl_div(b,mu)\n        # B library is used ONLY to define the evaluation target transform.\n        rate=np.divide(scale,lb,out=np.zeros_like(lb),where=lb>0)\n        y_b=np.log1p(b*rate[:,None]);secondary=(x_pred-y_b)**2\n        dev=losses.sum(axis=1,dtype=np.float64);sse=secondary.sum(axis=1,dtype=np.float64)\n    if not np.isfinite(dev).all() or not np.isfinite(sse).all():raise ValueError(\'Nonfinite metric; predictions/aggregation exceed numerical range\')\n    if (dev < -1e-9).any():raise AssertionError(\'Materially negative Poisson deviance\')\n    return pd.DataFrame(dict(deviance_sum=np.maximum(dev,0),normalized_sse_sum=sse,n_pairs=np.full(len(b),b.shape[1],dtype=np.int64),\n                             mu_floor_pairs=floored.sum(axis=1),negative_log_predictions=(pred<0).sum(axis=1) if prediction_type==\'log_normalized\' else np.zeros(len(b),dtype=np.int64)))\n\ndef relative_change(method_deviance,baseline_deviance,min_denominator=1e-12):\n    if not np.isfinite(min_denominator) or min_denominator<0:raise ValueError(\'Invalid relative denominator threshold\')\n    m=np.asarray(method_deviance,dtype=float);base=np.asarray(baseline_deviance,dtype=float)\n    if m.shape!=base.shape:raise ValueError(\'Relative metric shape mismatch\')\n    delta=m-base;valid=np.isfinite(m)&np.isfinite(base)&(base>min_denominator)\n    out=np.full(m.shape,np.nan);np.divide(100*delta,base,out=out,where=valid)\n    return out,np.where(valid,\'defined\',\'undefined_small_or_missing_baseline\')\n\ndef load_contract(cfg):\n    run=Path(cfg[\'run_path\']);prep=run/\'A_preprocessing\';contract=_json(prep/\'evaluation_pair_contract.json\')\n    cells_path=prep/contract[\'cells_file\'];genes_path=prep/contract[\'genes_file\']\n    if digest(cells_path)!=contract[\'cells_sha256\'] or digest(genes_path)!=contract[\'genes_sha256\']:raise ValueError(\'Frozen Step 2 pair files changed\')\n    cells=pd.read_parquet(cells_path).sort_values(\'matrix_row\').reset_index(drop=True);genes=pd.read_parquet(genes_path).sort_values(\'selected_col\')\n    if not np.array_equal(cells.matrix_row,np.arange(len(cells))) or not cells.cell_barcode.is_unique:raise ValueError(\'Cell row mapping mismatch\')\n    if not cells.evaluation_eligible.isin([True,False]).all():raise ValueError(\'Invalid frozen evaluation mask\')\n    if not np.array_equal(genes.selected_col,np.arange(len(genes))) or not genes.gene_idx.is_unique or not genes.gene_symbol.is_unique:raise ValueError(\'Gene column mapping mismatch\')\n    n=int(cells.evaluation_eligible.sum());g=len(genes)\n    if n<1 or g<1:raise ValueError(\'Frozen evaluation I must not be empty\')\n    if n!=contract[\'n_cells\'] or g!=contract[\'n_genes\'] or n*g!=contract[\'n_pairs\']:raise ValueError(\'Frozen pair counts mismatch\')\n    ident=_json(run/\'identity.json\')\n    if contract[\'q\']!=ident[\'q\']:raise ValueError(\'Splitting q mismatch\')\n    return run,contract,cells,genes,ident\n\ndef freeze_metrics(cfg):\n    run,contract,cells,genes,ident=load_contract(cfg);out=run/\'evaluation\';out.mkdir(exist_ok=True)\n    spec=dict(version=\'step3-fixed-evaluation-v1\',engine_sha256=ENGINE_SHA,pair_contract_sha256=digest(run/\'A_preprocessing/evaluation_pair_contract.json\'),\n        cells_sha256=contract[\'cells_sha256\'],genes_sha256=contract[\'genes_sha256\'],n_cells=contract[\'n_cells\'],n_genes=contract[\'n_genes\'],n_pairs=contract[\'n_pairs\'],\n        q=contract[\'q\'],normalization_scale=contract[\'normalization_scale\'],mu_floor=cfg[\'mu_floor\'],relative_min_denominator=cfg[\'relative_min_denominator\'],\n        baseline_method=\'CellOnly\',primary=\'sum 2*(B*log(B/mu)-B+mu)/n_pairs; zero B terms included\',\n        inverse_mapping=\'mu_B=expm1(max(log_prediction,0))*all_gene_L_A/scale*(1-q)/q; no B calibration\',\n        secondary=\'sum (raw_log_prediction-log1p(scale*B/all_gene_L_B))^2/n_pairs; target transform only\',\n        secondary_zero_B_library=\'zero target log values; same frozen pairs; no B-based exclusions\',\n        secondary_scope=\'manuscript log1p-normalized squared-error family on held-out B; not identical to old full-C targets\',\n        fixed_lower_bound=\'applied uniformly to count predictions only; no upper clipping\',target_role=\'final B evaluation only; no parameter selection\',\n        numpy=np.__version__,scipy=scipy.__version__)\n    if not np.isfinite(cfg[\'mu_floor\']) or cfg[\'mu_floor\']<=0 or not np.isfinite(cfg[\'relative_min_denominator\']) or cfg[\'relative_min_denominator\']<0:raise ValueError(\'Invalid metric floor/denominator\')\n    path=out/\'metric_contract.json\'\n    if path.exists() and _json(path)!=spec:raise ValueError(\'Metric contract already frozen differently; do not alter it after observing final B\')\n    save_json(path,spec)\n    group=[\'cell_line\',\'drug\',\'replicate\',\'dose_raw\',\'sgRNA_raw\',\'target_raw\',\'hash_plate\',\'hash_label_well\',\'library_id\']\n    group=[c for c in group if c in cells]\n    coverage=cells.loc[cells.evaluation_eligible].groupby(group,dropna=False,observed=True).size().rename(\'n_cells\').reset_index();coverage[\'n_pairs\']=coverage.n_cells*len(genes)\n    if int(coverage.n_pairs.sum())!=contract[\'n_pairs\']:raise AssertionError(\'Coverage mismatch\')\n    coverage.to_csv(out/\'evaluation_coverage.csv\',index=False)\n    template={\'method_name\':\'CellOnly\',\'prediction_type\':\'log_normalized\',\'prediction_folder\':\'FILL_AFTER_A_ONLY_FITTING\',\n        \'required_files\':[\'gene_idx.npy\',\'block_00000.npy\',\'block_00000_rows.npy\',\'one prediction+row_ids pair for every block\'],\n        \'training_provenance\':{\'method_input\':\'A only\',\'parameter_selection\':\'pre_fixed or separate_development\',\'final_B_used\':False,\'evidence_file\':\'path to reviewed fitting/config log\'},\n        \'notes\':\'npy is all retained rows in each count block by selected-gene order; row ids are global matrix_row; register_predictions verifies IDs and hashes\'}\n    save_json(out/\'prediction_manifest_template.json\',template)\n    return out,spec\n\ndef register_predictions(cfg,method_name,prediction_folder,prediction_type,training_provenance):\n    out,spec=freeze_metrics(cfg);run,contract,cells,genes,ident=load_contract(cfg)\n    if (out/\'target_access_lock.json\').exists():raise ValueError(\'Final B already accessed: cannot add or alter methods/predictions\')\n    if not method_name or Path(method_name).name!=method_name:raise ValueError(\'Method name must be one safe name\')\n    if prediction_type not in [\'log_normalized\',\'expected_B_counts\']:raise ValueError(\'Unsupported prediction type\')\n    if training_provenance.get(\'method_input\')!=\'A only\' or training_provenance.get(\'final_B_used\') is not False or training_provenance.get(\'parameter_selection\') not in [\'pre_fixed\',\'separate_development\']:raise ValueError(\'Reviewed A-only/pre-fixed or development-only provenance required\')\n    evidence=Path(training_provenance[\'evidence_file\'])\n    if not evidence.is_file():raise FileNotFoundError(\'Provide saved fitting/config evidence; declarations alone are not proof\')\n    folder=Path(prediction_folder).resolve();gene_file=folder/\'gene_idx.npy\'\n    ids=np.load(gene_file,allow_pickle=False)\n    if not np.array_equal(ids,genes.gene_idx.to_numpy()):raise ValueError(\'Prediction gene IDs/order differ from common I\')\n    records=[];size=ident[\'cells_per_block\']\n    for block in range((len(cells)+size-1)//size):\n        p=folder/f\'block_{block:05d}.npy\';rows=folder/f\'block_{block:05d}_rows.npy\';start=block*size;stop=min(start+size,len(cells))\n        x=np.load(p,mmap_mode=\'r\',allow_pickle=False);row_ids=np.load(rows,allow_pickle=False)\n        if x.shape!=(stop-start,len(genes)) or x.dtype.kind not in \'fiu\' or not np.array_equal(row_ids,np.arange(start,stop)):raise ValueError(\'Prediction shard shape/row IDs mismatch\')\n        count_folder=run/f\'block_{block:05d}\';count_receipt=_json(count_folder/\'receipt.json\')\n        source_hash=next(v[\'sha256\'] for v in count_receipt[\'files\'] if v[\'name\']==\'A_input.npz\')\n        if digest(count_folder/\'A_input.npz\')!=source_hash:raise ValueError(\'A input changed before prediction registration\')\n        input_A=sp.load_npz(count_folder/\'A_input.npz\');lib_A=np.asarray(input_A.sum(axis=1)).ravel().astype(float);del input_A\n        mask=cells.evaluation_eligible.iloc[start:stop].to_numpy(dtype=bool)\n        for lo in range(0,stop-start,cfg[\'batch_cells\']):\n            hi=min(lo+cfg[\'batch_cells\'],stop-start);chosen=mask[lo:hi]\n            if not chosen.any():continue\n            values=np.asarray(x[lo:hi][chosen],dtype=float);la=lib_A[lo:hi][chosen]\n            if not np.isfinite(values).all() or (la<=0).any():raise ValueError(\'Invalid evaluated prediction values/A exposure\')\n            if prediction_type==\'log_normalized\':mu=inverse_log1p_to_B(values,la,spec[\'q\'],spec[\'normalization_scale\'])\n            else:\n                if (values<0).any():raise ValueError(\'Negative expected counts\')\n                mu=values\n            if (mu>np.finfo(np.float64).max/(2*spec[\'n_pairs\'])).any():raise ValueError(\'Predictions exceed safe float64 aggregate range; no silent cap\')\n        records.append(dict(block=block,path=str(p),sha256=digest(p),rows_path=str(rows),rows_sha256=digest(rows)))\n    item=dict(method_name=method_name,prediction_type=prediction_type,metric_contract_sha256=digest(out/\'metric_contract.json\'),\n              genes_path=str(gene_file),genes_sha256=digest(gene_file),training_provenance=training_provenance,training_evidence_sha256=digest(evidence),blocks=records,\n              evidence_limitation=\'Provenance log requires human review; evaluator cannot prove how upstream model was fitted\')\n    registry=out/\'prediction_registry.json\';registered=_json(registry) if registry.exists() else {}\n    if method_name in registered and registered[method_name]!=item:raise ValueError(\'Registered method changed; preserve original registry and review before any final evaluation\')\n    registered[method_name]=item;save_json(registry,registered);return item\n\ndef _aggregate_cell_scores(frame,keys):\n    keys=[c for c in keys if c in frame];metrics=[\'deviance_sum\',\'normalized_sse_sum\',\'n_pairs\',\'mu_floor_pairs\',\'negative_log_predictions\']\n    grouped=frame.groupby(keys,dropna=False,observed=True)[metrics].sum().reset_index()\n    counts=frame.groupby(keys,dropna=False,observed=True).size().rename(\'n_cells\').reset_index()\n    return grouped.merge(counts,on=keys,validate=\'one_to_one\')\n\ndef final_evaluate(cfg):\n    """All registered predictions are frozen before target access; no fit callback."""\n    out,spec=freeze_metrics(cfg);run,contract,cells,genes,ident=load_contract(cfg)\n    registry=_json(out/\'prediction_registry.json\')\n    if \'CellOnly\' not in registry:raise ValueError(\'Actual CellOnly baseline predictions must be registered; no fabricated baseline\')\n    if cfg[\'batch_cells\']<1:raise ValueError(\'Invalid score batch size\')\n    snapshot=dict(metric_contract_sha256=digest(out/\'metric_contract.json\'),registry_sha256=digest(out/\'prediction_registry.json\'))\n    lock=out/\'target_access_lock.json\'\n    if lock.exists() and _json(lock)!=snapshot:raise ValueError(\'Cannot change predictions/metric after final target access\')\n    # Check every prediction before touching B.\n    for item in registry.values():\n        if item[\'metric_contract_sha256\']!=snapshot[\'metric_contract_sha256\']:raise ValueError(\'Registry metric contract mismatch\')\n        for record in item[\'blocks\']:\n            if digest(record[\'path\'])!=record[\'sha256\'] or digest(record[\'rows_path\'])!=record[\'rows_sha256\']:raise ValueError(\'Prediction cache changed\')\n        if digest(item[\'genes_path\'])!=item[\'genes_sha256\'] or digest(item[\'training_provenance\'][\'evidence_file\'])!=item[\'training_evidence_sha256\']:raise ValueError(\'Gene order or training evidence changed\')\n    save_json(lock,snapshot) # Computational retry allowed ONLY with the same frozen files.\n    selected=genes.gene_col.to_numpy(dtype=np.int64);size=ident[\'cells_per_block\'];pieces=[];rep_pieces=[];fine_pieces=[]\n    for block in range((len(cells)+size-1)//size):\n        folder=run/f\'block_{block:05d}\';receipt=_json(folder/\'receipt.json\');hashes={x[\'name\']:x[\'sha256\'] for x in receipt[\'files\']}\n        for name in [\'A_input.npz\',\'B_target.npz\']:\n            if digest(folder/name)!=hashes[name]:raise ValueError(\'Count shard changed\')\n        a=sp.load_npz(folder/\'A_input.npz\');la=np.asarray(a.sum(axis=1)).ravel().astype(float);del a\n        b=sp.load_npz(folder/\'B_target.npz\').tocsr()\n        if b.dtype.kind not in \'iu\' or (b.data<0).any():raise ValueError(\'Invalid held-out count shard\')\n        lb=np.asarray(b.sum(axis=1)).ravel().astype(float);b=b[:,selected]\n        start=block*size;stop=start+b.shape[0];meta=cells.iloc[start:stop].reset_index(drop=True)\n        if not np.array_equal(meta.matrix_row,np.arange(start,stop)):raise AssertionError(\'Shard row offset mismatch\')\n        arrays={method:np.load(item[\'blocks\'][block][\'path\'],mmap_mode=\'r\',allow_pickle=False) for method,item in registry.items()}\n        for lo in range(0,len(meta),cfg[\'batch_cells\']):\n            hi=min(lo+cfg[\'batch_cells\'],len(meta));keep=meta.evaluation_eligible.iloc[lo:hi].to_numpy(dtype=bool)\n            if not keep.any():continue\n            bs=b[lo:hi][keep];am=la[lo:hi][keep];bm=lb[lo:hi][keep];labels=meta.iloc[lo:hi].loc[keep].reset_index(drop=True)\n            for method,item in registry.items():\n                scored=score_rows(bs,np.asarray(arrays[method][lo:hi][keep]),am,bm,contract[\'q\'],contract[\'normalization_scale\'],spec[\'mu_floor\'],item[\'prediction_type\'])\n                row=pd.concat([labels,scored],axis=1);row[\'method\']=method\n                pieces.append(_aggregate_cell_scores(row,[\'method\',\'cell_line\',\'drug\']))\n                rep_pieces.append(_aggregate_cell_scores(row,[\'method\',\'cell_line\',\'drug\',\'replicate\']))\n                fine_pieces.append(_aggregate_cell_scores(row,[\'method\',\'cell_line\',\'drug\',\'replicate\',\'dose_raw\',\'sgRNA_raw\',\'target_raw\',\'hash_plate\',\'hash_label_well\',\'library_id\']))\n        print(f\'Evaluation block {block+1}/{(len(cells)+size-1)//size}\',flush=True)\n    def finish(parts,filename,keys,complete_contexts=False):\n        frame=pd.concat(parts,ignore_index=True)\n        group=[k for k in keys if k in frame];nums=[\'deviance_sum\',\'normalized_sse_sum\',\'n_pairs\',\'n_cells\',\'mu_floor_pairs\',\'negative_log_predictions\']\n        table=frame.groupby(group,dropna=False,observed=True)[nums].sum().reset_index()\n        if complete_contexts:\n            universe=pd.MultiIndex.from_product([list(registry),LINES,DRUGS],names=[\'method\',\'cell_line\',\'drug\']).to_frame(index=False)\n            table=universe.merge(table,on=group,how=\'left\');table[nums]=table[nums].fillna(0)\n        pairs=table.n_pairs.to_numpy(dtype=float)\n        table[\'poisson_deviance\']=np.divide(table.deviance_sum,pairs,out=np.full(len(table),np.nan),where=pairs>0)\n        table[\'normalized_MSE\']=np.divide(table.normalized_sse_sum,pairs,out=np.full(len(table),np.nan),where=pairs>0)\n        baseline=table.loc[table.method.eq(\'CellOnly\'),group[1:]+[\'poisson_deviance\',\'n_pairs\']].rename(columns={\'poisson_deviance\':\'CellOnly_deviance\',\'n_pairs\':\'CellOnly_n_pairs\'})\n        table=table.merge(baseline,on=group[1:],how=\'left\',validate=\'many_to_one\')\n        if not table.n_pairs.eq(table.CellOnly_n_pairs).all():raise AssertionError(\'Methods use different pair counts\')\n        table[\'absolute_deviance_difference\']=table.poisson_deviance-table.CellOnly_deviance\n        table[\'delta_percent\'],table[\'relative_status\']=relative_change(table.poisson_deviance,table.CellOnly_deviance,spec[\'relative_min_denominator\'])\n        table.to_csv(out/filename,index=False);return table\n    context=finish(pieces,\'context_scores.csv\',[\'method\',\'cell_line\',\'drug\'],True)\n    finish(rep_pieces,\'replicate_label_scores.csv\',[\'method\',\'cell_line\',\'drug\',\'replicate\'])\n    finish(fine_pieces,\'fine_condition_scores.csv\',[\'method\',\'cell_line\',\'drug\',\'replicate\',\'dose_raw\',\'sgRNA_raw\',\'target_raw\',\'hash_plate\',\'hash_label_well\',\'library_id\'])\n    for method,g in context.groupby(\'method\'):\n        if int(g.n_pairs.sum())!=contract[\'n_pairs\']:raise AssertionError(\'Full common I not scored\')\n    global_scores=context.groupby(\'method\',as_index=False)[[\'deviance_sum\',\'normalized_sse_sum\',\'n_pairs\']].sum()\n    global_scores[\'poisson_deviance\']=global_scores.deviance_sum/global_scores.n_pairs;global_scores[\'normalized_MSE\']=global_scores.normalized_sse_sum/global_scores.n_pairs\n    base=float(global_scores.loc[global_scores.method.eq(\'CellOnly\'),\'poisson_deviance\'].iloc[0]);global_scores[\'CellOnly_deviance\']=base\n    global_scores[\'absolute_deviance_difference\']=global_scores.poisson_deviance-base\n    global_scores[\'delta_percent\'],global_scores[\'relative_status\']=relative_change(global_scores.poisson_deviance,np.full(len(global_scores),base),spec[\'relative_min_denominator\'])\n    global_scores.to_csv(out/\'global_scores.csv\',index=False)\n    save_json(out/\'final_evaluation_summary.json\',dict(status=\'complete\',target_accessed=True,n_methods=len(registry),n_pairs_per_method=contract[\'n_pairs\'],snapshot=snapshot,\n        interpretation=\'Fixed predictive scoring criterion; not evidence that counts follow Poisson noise or independent biological replication\'))\n    return context,global_scores\n\n\ndef save_step3_setup(cfg,test_report,source_code):\n    if test_report.get(\'status\')!=\'PASS\':raise ValueError(\'Synthetic scoring tests must pass before freeze\')\n    out,spec=freeze_metrics(cfg);root=Path(cfg[\'project_root\'])\n    (root/\'src\').mkdir(exist_ok=True);code_path=root/\'src/evaluation.py\'\n    if code_path.exists() and digest(code_path)!=hashlib.sha256(source_code.encode()).hexdigest():\n        backup=code_path.with_name(\'evaluation_before_step3_\'+digest(code_path)[:12]+\'.py\')\n        if not backup.exists():backup.write_bytes(code_path.read_bytes())\n    code_path.write_text(source_code)\n    save_json(out/\'synthetic_test_report.json\',test_report)\n    summary=dict(status=\'metric_contract_fixed\',run_path=cfg[\'run_path\'],evaluation_path=str(out),engine_sha256=ENGINE_SHA,\n        synthetic_tests=\'PASS\',n_cells=spec[\'n_cells\'],n_genes=spec[\'n_genes\'],n_pairs=spec[\'n_pairs\'],\n        primary=\'held-out B mean Poisson deviance\',mu_floor=spec[\'mu_floor\'],inverse_mapping=spec[\'inverse_mapping\'],secondary=spec[\'secondary\'],\n        relative=\'100*(D_method-D_CellOnly)/D_CellOnly; negative is better; retain absolute values\',\n        target_accessed=(out/\'target_access_lock.json\').exists(),prediction_methods=list(_json(out/\'prediction_registry.json\')) if (out/\'prediction_registry.json\').exists() else [],\n        method_fitting=\'not performed by evaluator\',limitations=[\'Poisson scoring is not verification of Poisson noise\',\'biological replication unresolved\',\'fixed Step 2 C-based cell population conditioning persists\'])\n    (root/\'results/raw\').mkdir(parents=True,exist_ok=True);(root/\'configs\').mkdir(exist_ok=True)\n    save_json(root/\'results/raw/step3_summary.json\',summary);save_json(root/\'configs/evaluation_metric.json\',spec)\n    save_json(out/\'step3_config.json\',cfg)\n    try:commit=subprocess.check_output([\'git\',\'-C\',str(root),\'rev-parse\',\'HEAD\'],stderr=subprocess.DEVNULL,text=True).strip()\n    except (OSError,subprocess.CalledProcessError):commit=\'not_available\'\n    save_json(out/\'environment.json\',dict(python=sys.version,platform=platform.platform(),numpy=np.__version__,scipy=scipy.__version__,pandas=pd.__version__,git_commit=commit,seed=cfg[\'seed\']))\n    return summary\n'
"""Fixed count-level evaluator; fitting/calibration accepts no final target information."""
from pathlib import Path
import json, hashlib, sys, platform, subprocess
import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
from scipy.special import kl_div

ENGINE_SHA=hashlib.sha256(Path(__file__).read_bytes()).hexdigest() if '__file__' in globals() else None
LINES=['A172','T98G','U87MG'];DRUGS=['Vehicle','Lapatinib','Nintedanib','Trametinib','ZSTK474']

def digest(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()

def save_json(p,obj):
    p=Path(p);t=p.with_name(p.name+'.tmp');t.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str));t.replace(p)

def _fingerprint(obj):return hashlib.sha256(json.dumps(obj,sort_keys=True,separators=(',',':')).encode()).hexdigest()

def _json(p):return json.loads(Path(p).read_text())

def inverse_log1p_to_B(xhat,library_A,q,scale=10000):
    """No B-library fitting, no selected-gene renormalization, no upper clipping."""
    x=np.asarray(xhat,dtype=np.float64);lib=np.asarray(library_A,dtype=np.float64)
    if x.ndim!=2 or lib.shape!=(len(x),) or not np.isfinite(x).all() or not np.isfinite(lib).all() or (lib<=0).any():raise ValueError('Invalid prediction or positive-A-library inputs')
    if not 0<q<1 or not np.isfinite(scale) or scale<=0:raise ValueError('Invalid q/normalization scale')
    with np.errstate(over='raise',invalid='raise'):
        mu=np.expm1(np.maximum(x,0))*lib[:,None]/scale*(1-q)/q
    if not np.isfinite(mu).all():raise ValueError('Nonfinite count predictions; no hidden upper cap')
    return mu

def score_rows(B,prediction,library_A,library_B,q,scale,mu_floor,prediction_type='log_normalized'):
    """Mean is formed later with exact pair counts. Includes zero target entries."""
    b=B.toarray().astype(np.float64) if sp.issparse(B) else np.asarray(B,dtype=np.float64)
    pred=np.asarray(prediction,dtype=np.float64);la=np.asarray(library_A,dtype=np.float64);lb=np.asarray(library_B,dtype=np.float64)
    if b.ndim!=2 or pred.shape!=b.shape or la.shape!=(len(b),) or lb.shape!=(len(b),):raise ValueError('Evaluation shape mismatch')
    if not np.isfinite(b).all() or (b<0).any() or not np.equal(b,np.floor(b)).all():raise ValueError('B must be finite nonnegative integer counts')
    if not np.isfinite(pred).all() or not np.isfinite(lb).all() or (lb<0).any() or (b.sum(axis=1)>lb+1e-8).any():raise ValueError('Invalid predictions or all-gene B library')
    if not np.isfinite(la).all() or (la<=0).any() or not 0<q<1 or scale<=0 or not np.isfinite(mu_floor) or mu_floor<=0:raise ValueError('Invalid fixed scoring configuration')
    if prediction_type=='log_normalized':
        mu=inverse_log1p_to_B(pred,la,q,scale);x_pred=pred
    elif prediction_type=='expected_B_counts':
        if (pred<0).any():raise ValueError('Expected-count predictions must be nonnegative')
        mu=pred.copy();x_pred=np.log1p(scale*mu/(la[:,None]*(1-q)/q))
    else:raise ValueError('Unknown prediction type')
    floored=mu<mu_floor;mu=np.maximum(mu,mu_floor)
    with np.errstate(over='raise',invalid='raise'):
        losses=2*kl_div(b,mu)

        rate=np.divide(scale,lb,out=np.zeros_like(lb),where=lb>0)
        y_b=np.log1p(b*rate[:,None]);secondary=(x_pred-y_b)**2
        dev=losses.sum(axis=1,dtype=np.float64);sse=secondary.sum(axis=1,dtype=np.float64)
    if not np.isfinite(dev).all() or not np.isfinite(sse).all():raise ValueError('Nonfinite metric; predictions/aggregation exceed numerical range')
    if (dev < -1e-9).any():raise AssertionError('Materially negative Poisson deviance')
    return pd.DataFrame(dict(deviance_sum=np.maximum(dev,0),normalized_sse_sum=sse,n_pairs=np.full(len(b),b.shape[1],dtype=np.int64),
                             mu_floor_pairs=floored.sum(axis=1),negative_log_predictions=(pred<0).sum(axis=1) if prediction_type=='log_normalized' else np.zeros(len(b),dtype=np.int64)))

def relative_change(method_deviance,baseline_deviance,min_denominator=1e-12):
    if not np.isfinite(min_denominator) or min_denominator<0:raise ValueError('Invalid relative denominator threshold')
    m=np.asarray(method_deviance,dtype=float);base=np.asarray(baseline_deviance,dtype=float)
    if m.shape!=base.shape:raise ValueError('Relative metric shape mismatch')
    delta=m-base;valid=np.isfinite(m)&np.isfinite(base)&(base>min_denominator)
    out=np.full(m.shape,np.nan);np.divide(100*delta,base,out=out,where=valid)
    return out,np.where(valid,'defined','undefined_small_or_missing_baseline')

def load_contract(cfg):
    run=Path(cfg['run_path']);prep=run/'A_preprocessing';contract=_json(prep/'evaluation_pair_contract.json')
    cells_path=prep/contract['cells_file'];genes_path=prep/contract['genes_file']
    if digest(cells_path)!=contract['cells_sha256'] or digest(genes_path)!=contract['genes_sha256']:raise ValueError('Frozen Step 2 pair files changed')
    cells=pd.read_parquet(cells_path).sort_values('matrix_row').reset_index(drop=True);genes=pd.read_parquet(genes_path).sort_values('selected_col')
    if not np.array_equal(cells.matrix_row,np.arange(len(cells))) or not cells.cell_barcode.is_unique:raise ValueError('Cell row mapping mismatch')
    if not cells.evaluation_eligible.isin([True,False]).all():raise ValueError('Invalid frozen evaluation mask')
    if not np.array_equal(genes.selected_col,np.arange(len(genes))) or not genes.gene_idx.is_unique or not genes.gene_symbol.is_unique:raise ValueError('Gene column mapping mismatch')
    n=int(cells.evaluation_eligible.sum());g=len(genes)
    if n<1 or g<1:raise ValueError('Frozen evaluation I must not be empty')
    if n!=contract['n_cells'] or g!=contract['n_genes'] or n*g!=contract['n_pairs']:raise ValueError('Frozen pair counts mismatch')
    ident=_json(run/'identity.json')
    if contract['q']!=ident['q']:raise ValueError('Splitting q mismatch')
    return run,contract,cells,genes,ident

def freeze_metrics(cfg):
    run,contract,cells,genes,ident=load_contract(cfg);out=run/'evaluation';out.mkdir(exist_ok=True)
    spec=dict(version='step3-fixed-evaluation-v1',engine_sha256=ENGINE_SHA,pair_contract_sha256=digest(run/'A_preprocessing/evaluation_pair_contract.json'),
        cells_sha256=contract['cells_sha256'],genes_sha256=contract['genes_sha256'],n_cells=contract['n_cells'],n_genes=contract['n_genes'],n_pairs=contract['n_pairs'],
        q=contract['q'],normalization_scale=contract['normalization_scale'],mu_floor=cfg['mu_floor'],relative_min_denominator=cfg['relative_min_denominator'],
        baseline_method='CellOnly',primary='sum 2*(B*log(B/mu)-B+mu)/n_pairs; zero B terms included',
        inverse_mapping='mu_B=expm1(max(log_prediction,0))*all_gene_L_A/scale*(1-q)/q; no B calibration',
        secondary='sum (raw_log_prediction-log1p(scale*B/all_gene_L_B))^2/n_pairs; target transform only',
        secondary_zero_B_library='zero target log values; same frozen pairs; no B-based exclusions',
        secondary_scope='manuscript log1p-normalized squared-error family on held-out B; not identical to old full-C targets',
        fixed_lower_bound='applied uniformly to count predictions only; no upper clipping',target_role='final B evaluation only; no parameter selection',
        numpy=np.__version__,scipy=scipy.__version__)
    if not np.isfinite(cfg['mu_floor']) or cfg['mu_floor']<=0 or not np.isfinite(cfg['relative_min_denominator']) or cfg['relative_min_denominator']<0:raise ValueError('Invalid metric floor/denominator')
    path=out/'metric_contract.json'
    if path.exists() and _json(path)!=spec:raise ValueError('Metric contract already frozen differently; do not alter it after observing final B')
    save_json(path,spec)
    group=['cell_line','drug','replicate','dose_raw','sgRNA_raw','target_raw','hash_plate','hash_label_well','library_id']
    group=[c for c in group if c in cells]
    coverage=cells.loc[cells.evaluation_eligible].groupby(group,dropna=False,observed=True).size().rename('n_cells').reset_index();coverage['n_pairs']=coverage.n_cells*len(genes)
    if int(coverage.n_pairs.sum())!=contract['n_pairs']:raise AssertionError('Coverage mismatch')
    coverage.to_csv(out/'evaluation_coverage.csv',index=False)
    template={'method_name':'CellOnly','prediction_type':'log_normalized','prediction_folder':'FILL_AFTER_A_ONLY_FITTING',
        'required_files':['gene_idx.npy','block_00000.npy','block_00000_rows.npy','one prediction+row_ids pair for every block'],
        'training_provenance':{'method_input':'A only','parameter_selection':'pre_fixed or separate_development','final_B_used':False,'evidence_file':'path to reviewed fitting/config log'},
        'notes':'npy is all retained rows in each count block by selected-gene order; row ids are global matrix_row; register_predictions verifies IDs and hashes'}
    save_json(out/'prediction_manifest_template.json',template)
    return out,spec

def register_predictions(cfg,method_name,prediction_folder,prediction_type,training_provenance):
    out,spec=freeze_metrics(cfg);run,contract,cells,genes,ident=load_contract(cfg)
    if (out/'target_access_lock.json').exists():raise ValueError('Final B already accessed: cannot add or alter methods/predictions')
    if not method_name or Path(method_name).name!=method_name:raise ValueError('Method name must be one safe name')
    if prediction_type not in ['log_normalized','expected_B_counts']:raise ValueError('Unsupported prediction type')
    if training_provenance.get('method_input')!='A only' or training_provenance.get('final_B_used') is not False or training_provenance.get('parameter_selection') not in ['pre_fixed','separate_development']:raise ValueError('Reviewed A-only/pre-fixed or development-only provenance required')
    evidence=Path(training_provenance['evidence_file'])
    if not evidence.is_file():raise FileNotFoundError('Provide saved fitting/config evidence; declarations alone are not proof')
    folder=Path(prediction_folder).resolve();gene_file=folder/'gene_idx.npy'
    ids=np.load(gene_file,allow_pickle=False)
    if not np.array_equal(ids,genes.gene_idx.to_numpy()):raise ValueError('Prediction gene IDs/order differ from common I')
    records=[];size=ident['cells_per_block']
    for block in range((len(cells)+size-1)//size):
        p=folder/f'block_{block:05d}.npy';rows=folder/f'block_{block:05d}_rows.npy';start=block*size;stop=min(start+size,len(cells))
        x=np.load(p,mmap_mode='r',allow_pickle=False);row_ids=np.load(rows,allow_pickle=False)
        if x.shape!=(stop-start,len(genes)) or x.dtype.kind not in 'fiu' or not np.array_equal(row_ids,np.arange(start,stop)):raise ValueError('Prediction shard shape/row IDs mismatch')
        count_folder=run/f'block_{block:05d}';count_receipt=_json(count_folder/'receipt.json')
        source_hash=next(v['sha256'] for v in count_receipt['files'] if v['name']=='A_input.npz')
        if digest(count_folder/'A_input.npz')!=source_hash:raise ValueError('A input changed before prediction registration')
        input_A=sp.load_npz(count_folder/'A_input.npz');lib_A=np.asarray(input_A.sum(axis=1)).ravel().astype(float);del input_A
        mask=cells.evaluation_eligible.iloc[start:stop].to_numpy(dtype=bool)
        for lo in range(0,stop-start,cfg['batch_cells']):
            hi=min(lo+cfg['batch_cells'],stop-start);chosen=mask[lo:hi]
            if not chosen.any():continue
            values=np.asarray(x[lo:hi][chosen],dtype=float);la=lib_A[lo:hi][chosen]
            if not np.isfinite(values).all() or (la<=0).any():raise ValueError('Invalid evaluated prediction values/A exposure')
            if prediction_type=='log_normalized':mu=inverse_log1p_to_B(values,la,spec['q'],spec['normalization_scale'])
            else:
                if (values<0).any():raise ValueError('Negative expected counts')
                mu=values
            if (mu>np.finfo(np.float64).max/(2*spec['n_pairs'])).any():raise ValueError('Predictions exceed safe float64 aggregate range; no silent cap')
        records.append(dict(block=block,path=str(p),sha256=digest(p),rows_path=str(rows),rows_sha256=digest(rows)))
    item=dict(method_name=method_name,prediction_type=prediction_type,metric_contract_sha256=digest(out/'metric_contract.json'),
              genes_path=str(gene_file),genes_sha256=digest(gene_file),training_provenance=training_provenance,training_evidence_sha256=digest(evidence),blocks=records,
              evidence_limitation='Provenance log requires human review; evaluator cannot prove how upstream model was fitted')
    registry=out/'prediction_registry.json';registered=_json(registry) if registry.exists() else {}
    if method_name in registered and registered[method_name]!=item:raise ValueError('Registered method changed; preserve original registry and review before any final evaluation')
    registered[method_name]=item;save_json(registry,registered);return item

def _aggregate_cell_scores(frame,keys):
    keys=[c for c in keys if c in frame];metrics=['deviance_sum','normalized_sse_sum','n_pairs','mu_floor_pairs','negative_log_predictions']
    grouped=frame.groupby(keys,dropna=False,observed=True)[metrics].sum().reset_index()
    counts=frame.groupby(keys,dropna=False,observed=True).size().rename('n_cells').reset_index()
    return grouped.merge(counts,on=keys,validate='one_to_one')

def final_evaluate(cfg):
    """All registered predictions are frozen before target access; no fit callback."""
    out,spec=freeze_metrics(cfg);run,contract,cells,genes,ident=load_contract(cfg)
    registry=_json(out/'prediction_registry.json')
    if 'CellOnly' not in registry:raise ValueError('Actual CellOnly baseline predictions must be registered; no fabricated baseline')
    if cfg['batch_cells']<1:raise ValueError('Invalid score batch size')
    snapshot=dict(metric_contract_sha256=digest(out/'metric_contract.json'),registry_sha256=digest(out/'prediction_registry.json'))
    lock=out/'target_access_lock.json'
    if lock.exists() and _json(lock)!=snapshot:raise ValueError('Cannot change predictions/metric after final target access')
    # Check every prediction before touching B.
    for item in registry.values():
        if item['metric_contract_sha256']!=snapshot['metric_contract_sha256']:raise ValueError('Registry metric contract mismatch')
        for record in item['blocks']:
            if digest(record['path'])!=record['sha256'] or digest(record['rows_path'])!=record['rows_sha256']:raise ValueError('Prediction cache changed')
        if digest(item['genes_path'])!=item['genes_sha256'] or digest(item['training_provenance']['evidence_file'])!=item['training_evidence_sha256']:raise ValueError('Gene order or training evidence changed')
    save_json(lock,snapshot) # Computational retry allowed ONLY with the same frozen files.
    selected=genes.gene_col.to_numpy(dtype=np.int64);size=ident['cells_per_block'];pieces=[];rep_pieces=[];fine_pieces=[]
    for block in range((len(cells)+size-1)//size):
        folder=run/f'block_{block:05d}';receipt=_json(folder/'receipt.json');hashes={x['name']:x['sha256'] for x in receipt['files']}
        for name in ['A_input.npz','B_target.npz']:
            if digest(folder/name)!=hashes[name]:raise ValueError('Count shard changed')
        a=sp.load_npz(folder/'A_input.npz');la=np.asarray(a.sum(axis=1)).ravel().astype(float);del a
        b=sp.load_npz(folder/'B_target.npz').tocsr()
        if b.dtype.kind not in 'iu' or (b.data<0).any():raise ValueError('Invalid held-out count shard')
        lb=np.asarray(b.sum(axis=1)).ravel().astype(float);b=b[:,selected]
        start=block*size;stop=start+b.shape[0];meta=cells.iloc[start:stop].reset_index(drop=True)
        if not np.array_equal(meta.matrix_row,np.arange(start,stop)):raise AssertionError('Shard row offset mismatch')
        arrays={method:np.load(item['blocks'][block]['path'],mmap_mode='r',allow_pickle=False) for method,item in registry.items()}
        for lo in range(0,len(meta),cfg['batch_cells']):
            hi=min(lo+cfg['batch_cells'],len(meta));keep=meta.evaluation_eligible.iloc[lo:hi].to_numpy(dtype=bool)
            if not keep.any():continue
            bs=b[lo:hi][keep];am=la[lo:hi][keep];bm=lb[lo:hi][keep];labels=meta.iloc[lo:hi].loc[keep].reset_index(drop=True)
            for method,item in registry.items():
                scored=score_rows(bs,np.asarray(arrays[method][lo:hi][keep]),am,bm,contract['q'],contract['normalization_scale'],spec['mu_floor'],item['prediction_type'])
                row=pd.concat([labels,scored],axis=1);row['method']=method
                pieces.append(_aggregate_cell_scores(row,['method','cell_line','drug']))
                rep_pieces.append(_aggregate_cell_scores(row,['method','cell_line','drug','replicate']))
                fine_pieces.append(_aggregate_cell_scores(row,['method','cell_line','drug','replicate','dose_raw','sgRNA_raw','target_raw','hash_plate','hash_label_well','library_id']))
        print(f'Evaluation block {block+1}/{(len(cells)+size-1)//size}',flush=True)
    def finish(parts,filename,keys,complete_contexts=False):
        frame=pd.concat(parts,ignore_index=True)
        group=[k for k in keys if k in frame];nums=['deviance_sum','normalized_sse_sum','n_pairs','n_cells','mu_floor_pairs','negative_log_predictions']
        table=frame.groupby(group,dropna=False,observed=True)[nums].sum().reset_index()
        if complete_contexts:
            universe=pd.MultiIndex.from_product([list(registry),LINES,DRUGS],names=['method','cell_line','drug']).to_frame(index=False)
            table=universe.merge(table,on=group,how='left');table[nums]=table[nums].fillna(0)
        pairs=table.n_pairs.to_numpy(dtype=float)
        table['poisson_deviance']=np.divide(table.deviance_sum,pairs,out=np.full(len(table),np.nan),where=pairs>0)
        table['normalized_MSE']=np.divide(table.normalized_sse_sum,pairs,out=np.full(len(table),np.nan),where=pairs>0)
        baseline=table.loc[table.method.eq('CellOnly'),group[1:]+['poisson_deviance','n_pairs']].rename(columns={'poisson_deviance':'CellOnly_deviance','n_pairs':'CellOnly_n_pairs'})
        table=table.merge(baseline,on=group[1:],how='left',validate='many_to_one')
        if not table.n_pairs.eq(table.CellOnly_n_pairs).all():raise AssertionError('Methods use different pair counts')
        table['absolute_deviance_difference']=table.poisson_deviance-table.CellOnly_deviance
        table['delta_percent'],table['relative_status']=relative_change(table.poisson_deviance,table.CellOnly_deviance,spec['relative_min_denominator'])
        table.to_csv(out/filename,index=False);return table
    context=finish(pieces,'context_scores.csv',['method','cell_line','drug'],True)
    finish(rep_pieces,'replicate_label_scores.csv',['method','cell_line','drug','replicate'])
    finish(fine_pieces,'fine_condition_scores.csv',['method','cell_line','drug','replicate','dose_raw','sgRNA_raw','target_raw','hash_plate','hash_label_well','library_id'])
    for method,g in context.groupby('method'):
        if int(g.n_pairs.sum())!=contract['n_pairs']:raise AssertionError('Full common I not scored')
    global_scores=context.groupby('method',as_index=False)[['deviance_sum','normalized_sse_sum','n_pairs']].sum()
    global_scores['poisson_deviance']=global_scores.deviance_sum/global_scores.n_pairs;global_scores['normalized_MSE']=global_scores.normalized_sse_sum/global_scores.n_pairs
    base=float(global_scores.loc[global_scores.method.eq('CellOnly'),'poisson_deviance'].iloc[0]);global_scores['CellOnly_deviance']=base
    global_scores['absolute_deviance_difference']=global_scores.poisson_deviance-base
    global_scores['delta_percent'],global_scores['relative_status']=relative_change(global_scores.poisson_deviance,np.full(len(global_scores),base),spec['relative_min_denominator'])
    global_scores.to_csv(out/'global_scores.csv',index=False)
    save_json(out/'final_evaluation_summary.json',dict(status='complete',target_accessed=True,n_methods=len(registry),n_pairs_per_method=contract['n_pairs'],snapshot=snapshot,
        interpretation='Fixed predictive scoring criterion; not evidence that counts follow Poisson noise or independent biological replication'))
    return context,global_scores

def save_step3_setup(cfg,test_report,source_code):
    if test_report.get('status')!='PASS':raise ValueError('Synthetic scoring tests must pass before freeze')
    out,spec=freeze_metrics(cfg);root=Path(cfg['project_root'])
    (root/'src').mkdir(exist_ok=True);code_path=root/'src/evaluation.py'
    if code_path.exists() and digest(code_path)!=hashlib.sha256(source_code.encode()).hexdigest():
        backup=code_path.with_name('evaluation_before_step3_'+digest(code_path)[:12]+'.py')
        if not backup.exists():backup.write_bytes(code_path.read_bytes())
    code_path.write_text(source_code)
    save_json(out/'synthetic_test_report.json',test_report)
    summary=dict(status='metric_contract_fixed',run_path=cfg['run_path'],evaluation_path=str(out),engine_sha256=ENGINE_SHA,
        synthetic_tests='PASS',n_cells=spec['n_cells'],n_genes=spec['n_genes'],n_pairs=spec['n_pairs'],
        primary='held-out B mean Poisson deviance',mu_floor=spec['mu_floor'],inverse_mapping=spec['inverse_mapping'],secondary=spec['secondary'],
        relative='100*(D_method-D_CellOnly)/D_CellOnly; negative is better; retain absolute values',
        target_accessed=(out/'target_access_lock.json').exists(),prediction_methods=list(_json(out/'prediction_registry.json')) if (out/'prediction_registry.json').exists() else [],
        method_fitting='not performed by evaluator',limitations=['Poisson scoring is not verification of Poisson noise','biological replication unresolved','fixed Step 2 C-based cell population conditioning persists'])
    (root/'results/raw').mkdir(parents=True,exist_ok=True);(root/'configs').mkdir(exist_ok=True)
    save_json(root/'results/raw/step3_summary.json',summary);save_json(root/'configs/evaluation_metric.json',spec)
    save_json(out/'step3_config.json',cfg)
    try:commit=subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],stderr=subprocess.DEVNULL,text=True).strip()
    except (OSError,subprocess.CalledProcessError):commit='not_available'
    save_json(out/'environment.json',dict(python=sys.version,platform=platform.platform(),numpy=np.__version__,scipy=scipy.__version__,pandas=pd.__version__,git_commit=commit,seed=cfg['seed']))
    return summary

ENGINE_SHA = 'ef49f6084f7df1e18a2c003a78f5baddc9cde8dca56fcb1a2b0f8d814a5a8628'


In [3]:
import math
from sklearn.metrics import mean_poisson_deviance
def rejects(fn):
    try:
        fn()
    except (ValueError, AssertionError, FloatingPointError):
        return
    raise AssertionError('Expected rejection')

def oracle(b, mu, floor):
    mu = np.maximum(mu, floor)
    v = np.zeros_like(mu, dtype=float)
    for idx in np.ndindex(b.shape):
        y = b[idx]
        u = mu[idx]
        v[idx] = 2 * (y * math.log(y / u) - y + u) if y else 2 * u
    return v

def self_tests():
    rng = np.random.default_rng(9)
    counts = rng.poisson(4, (13, 7))
    counts[0] = 0
    lib = counts.sum(1).astype(float) + 3
    y = np.log1p(10000 * counts / lib[:, None])
    for q in [0.25, 0.5, 0.8]:
        assert np.allclose(inverse_log1p_to_B(y, lib, q), counts * (1 - q) / q)
    b = rng.poisson(5, (13, 7))
    b[0] = 0
    lb = b.sum(1).astype(float)
    mu = rng.uniform(0, 9, b.shape)
    mu[1, 2] = 0
    floor = 1e-08
    got = score_rows(sp.csr_matrix(b), mu, lib, lb, 0.5, 10000, floor, 'expected_B_counts')
    ref = oracle(b, mu, floor)
    assert np.allclose(got.deviance_sum, ref.sum(1))
    assert np.allclose(got.deviance_sum.sum() / b.size, mean_poisson_deviance(b.ravel(), np.maximum(mu, floor).ravel()))
    parts = [score_rows(b[i:i + 4], mu[i:i + 4], lib[i:i + 4], lb[i:i + 4], 0.5, 10000, floor, 'expected_B_counts') for i in range(0, len(b), 4)]
    assert np.allclose(pd.concat(parts).deviance_sum.to_numpy(), got.deviance_sum)
    target = np.zeros_like(b, dtype=float)
    positive = lb > 0
    target[positive] = np.log1p(10000 * b[positive] / lb[positive, None])
    x = np.log1p(10000 * mu / lib[:, None])
    assert np.allclose(got.normalized_sse_sum, ((x - target) ** 2).sum(1))
    zeros = score_rows(np.zeros((1, 3), dtype=int), np.zeros((1, 3)), np.array([1]), np.array([0]), 0.5, 10000, floor)
    assert np.allclose(zeros.deviance_sum.iloc[0], 6 * floor) and zeros.n_pairs.iloc[0] == 3 and (zeros.normalized_sse_sum.iloc[0] == 0)
    negative = score_rows(np.array([[0, 1]]), np.array([[-1.0, -2.0]]), np.array([5]), np.array([1]), 0.5, 10000, floor)
    assert negative.negative_log_predictions.iloc[0] == 2 and negative.mu_floor_pairs.iloc[0] == 2
    rel, status = relative_change(np.array([0.5, 2, 0]), np.array([1.0, 1.0, 0]))
    assert np.allclose(rel[:2], [-50, 100]) and np.isnan(rel[2])
    rejects(lambda: inverse_log1p_to_B([[np.nan]], [1], 0.5))
    rejects(lambda: inverse_log1p_to_B([[1000]], [1], 0.5))
    rejects(lambda: inverse_log1p_to_B([[1]], [0], 0.5))
    rejects(lambda: score_rows([[0.5]], [[1]], [1], [1], 0.5, 10000, floor))
    rejects(lambda: score_rows([[1]], [[-1]], [1], [1], 0.5, 10000, floor, 'expected_B_counts'))
    rejects(lambda: score_rows([[2]], [[1]], [1], [1], 0.5, 10000, floor))
    pred = inverse_log1p_to_B(y, lib, 0.5)
    b[:] = 100
    assert np.array_equal(pred, inverse_log1p_to_B(y, lib, 0.5))
    for observations in [rng.poisson(4, 50000), rng.poisson(rng.gamma(2, 2, 50000))]:
        scores = [float((2 * kl_div(observations, mean)).mean()) for mean in [2, 4, 8]]
        assert scores[1] < scores[0] and scores[1] < scores[2]
    return {'status': 'PASS', 'checks': ['dense independent oracle', 'sklearn equivalence', 'sparse/dense batch equivalence', 'zero B / all-zero rows', 'nonzero mu on zero target counted', 'q scaling', 'negative log clipping diagnostics', 'B-free calibration', 'NaN/overflow/shape/count validation', 'relative sign and zero denominator', 'conditional-mean ranking under Poisson and overdispersed synthetic counts']}
TEST_REPORT = self_tests()
print(json.dumps(TEST_REPORT,ensure_ascii=False,indent=2))


{
  "status": "PASS",
  "checks": [
    "dense independent oracle",
    "sklearn equivalence",
    "sparse/dense batch equivalence",
    "zero B / all-zero rows",
    "nonzero mu on zero target counted",
    "q scaling",
    "negative log clipping diagnostics",
    "B-free calibration",
    "NaN/overflow/shape/count validation",
    "relative sign and zero denominator",
    "conditional-mean ranking under Poisson and overdispersed synthetic counts"
  ]
}


In [4]:
from IPython.display import display
step3_summary = save_step3_setup(CONFIG, TEST_REPORT, SOURCE_CODE)
print(json.dumps(step3_summary,ensure_ascii=False,indent=2))
EVALUATION = Path(CONFIG['run_path'])/'evaluation'
display(pd.read_csv(EVALUATION/'evaluation_coverage.csv').head(15))
print('Metric contract:', EVALUATION/'metric_contract.json')


{
  "status": "metric_contract_fixed",
  "run_path": "/home/featurize/work/biological_prior_validation/data/splits/step2_q05_seed20261008",
  "evaluation_path": "/home/featurize/work/biological_prior_validation/data/splits/step2_q05_seed20261008/evaluation",
  "engine_sha256": "ef49f6084f7df1e18a2c003a78f5baddc9cde8dca56fcb1a2b0f8d814a5a8628",
  "synthetic_tests": "PASS",
  "n_cells": 989299,
  "n_genes": 2500,
  "n_pairs": 2473247500,
  "primary": "held-out B mean Poisson deviance",
  "mu_floor": 1e-08,
  "inverse_mapping": "mu_B=expm1(max(log_prediction,0))*all_gene_L_A/scale*(1-q)/q; no B calibration",
  "secondary": "sum (raw_log_prediction-log1p(scale*B/all_gene_L_B))^2/n_pairs; target transform only",
  "relative": "100*(D_method-D_CellOnly)/D_CellOnly; negative is better; retain absolute values",
  "target_accessed": false,
  "prediction_methods": [],
  "method_fitting": "not performed by evaluator",
  "limitations": [
    "Poisson scoring is not verification of Poisson noise",


,cell_line,drug,replicate,dose_raw,sgRNA_raw,target_raw,hash_plate,hash_label_well,library_id,n_cells,n_pairs
0,A172,Lapatinib,replicate_1,1,AAK1_1,AAK1,10,E4,sci3_CRISPRi_kinome,1,2500
1,A172,Lapatinib,replicate_1,1,AAK1_1,AAK1,8,E6,sci3_CRISPRi_kinome,1,2500
2,A172,Lapatinib,replicate_1,1,AAK1_1,AAK1,9,E5,sci3_CRISPRi_kinome,1,2500
3,A172,Lapatinib,replicate_1,1,AAK1_1,AAK1,9,G6,sci3_CRISPRi_kinome,1,2500
4,A172,Lapatinib,replicate_1,1,"AAK1_1,BMPR1B_5,MAPK13_6","AAK1,BMPR1B,MAPK13",9,F5,sci3_CRISPRi_kinome,1,2500
5,A172,Lapatinib,replicate_1,1,AAK1_2,AAK1,10,F5,sci3_CRISPRi_kinome,1,2500
6,A172,Lapatinib,replicate_1,1,AAK1_2,AAK1,9,E5,sci3_CRISPRi_kinome,1,2500
7,A172,Lapatinib,replicate_1,1,AAK1_2,AAK1,9,G5,sci3_CRISPRi_kinome,1,2500
8,A172,Lapatinib,replicate_1,1,"AAK1_2,MAST4_3,RIPK2_2","AAK1,MAST4,RIPK2",11,G6,sci3_CRISPRi_kinome,1,2500
9,A172,Lapatinib,replicate_1,1,AAK1_3,AAK1,10,E4,sci3_CRISPRi_kinome,2,5000


Metric contract: /home/featurize/work/biological_prior_validation/data/splits/step2_q05_seed20261008/evaluation/metric_contract.json


In [5]:
for item in METHOD_SPECS:
    register_predictions(
        CONFIG,
        method_name=item['method_name'],
        prediction_folder=item['prediction_folder'],
        prediction_type=item['prediction_type'],
        training_provenance=item['training_provenance'],
    )

if RUN_FINAL_EVALUATION:
    if not (EVALUATION/'prediction_registry.json').is_file():
        raise FileNotFoundError('First generate and register the real CellOnly and other method predictions; do not substitute with sample data.')
    context_scores, global_scores = final_evaluate(CONFIG)
    display(global_scores)
    display(context_scores)

    step3_summary = save_step3_setup(CONFIG, TEST_REPORT, SOURCE_CODE)
else:
    print('Step 3 The contract evaluation and scoring code have been fixed; the real final B has not yet been used for the main method scoring.')


Step 3 The contract evaluation and scoring code have been fixed; the real final B has not yet been used for the main method scoring.
